# GraphTrust verified large run: Global Hybrid

Run every cell from top to bottom in a **CPU high-RAM** Colab runtime. The notebook persists the generated graph and immutable run under Google Drive, so reconnecting and running again resumes instead of discarding verified work. The frozen source/target budgets in `configs/large.yaml` make this a bounded scalability run, not an exhaustive-recall experiment.


## 0. Settings and persistent workspace

In [ ]:
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, files

PROFILE = 'global_hybrid'
SEED = 2750159
REPO = 'https://github.com/sronters/graph.git'
BRANCH = 'main'  # PR #2 must be merged before this run.
ROOT = Path('/content/graphtrust')
drive.mount('/content/drive')
PERSIST = Path('/content/drive/MyDrive/GraphTrustLarge') / PROFILE / str(SEED)
DATA_ROOT = PERSIST / 'data'
OUTPUT_ROOT = PERSIST / 'artifacts'
PERSIST.mkdir(parents=True, exist_ok=True)
print({'profile': PROFILE, 'seed': SEED, 'persistent_workspace': str(PERSIST)})


## 1. Clone the final source and install the locked environment

In [ ]:
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', '--detach', f'origin/{BRANCH}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'sync', '--frozen', '--all-extras'], check=True)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print({'git_commit': COMMIT, 'python': sys.version, 'platform': platform.platform()})


## 2. Fail-fast capacity gate and smoke test

In [ ]:
import psutil

available_ram_gib = psutil.virtual_memory().available / 2**30
drive_free_gib = shutil.disk_usage(PERSIST).free / 2**30
assert os.getenv('COLAB_RELEASE_TAG'), 'This must run inside Google Colab.'
assert available_ram_gib >= 20, f'Choose a high-RAM runtime; only {available_ram_gib:.1f} GiB is available.'
assert drive_free_gib >= 15, f'Free at least 15 GiB in Google Drive; only {drive_free_gib:.1f} GiB is available.'
subprocess.run(['uv', 'run', 'graphtrust', 'generate', '--profile', PROFILE, '--scale', 'large', '--seed', str(SEED), '--variants', 'injected_mixed', '--dry-run'], check=True)
subprocess.run(['uv', 'run', 'pytest', '-q', 'tests/integration/test_experiment_runner.py'], check=True)
print({'available_ram_gib': round(available_ram_gib, 1), 'drive_free_gib': round(drive_free_gib, 1), 'smoke_test': 'passed'})


## 3. Generate or reuse the deterministic 2.5-million-edge graph

In [ ]:
dataset = DATA_ROOT / PROFILE / 'large' / str(SEED) / 'injected_mixed'
if not (dataset / 'checksums.sha256').exists():
    subprocess.run(['uv', 'run', 'graphtrust', 'generate', '--profile', PROFILE, '--scale', 'large', '--seed', str(SEED), '--variants', 'injected_mixed', '--output-root', str(DATA_ROOT)], check=True)
subprocess.run(['uv', 'run', 'graphtrust', 'validate-data', '--dataset', str(dataset)], check=True)
print({'dataset': str(dataset), 'status': 'checksum-verified'})


## 4. Run or resume the immutable bounded analysis

In [ ]:
subprocess.run(['uv', 'run', 'python', 'scripts/run_large_profiles.py', '--profile', PROFILE, '--seed', str(SEED), '--data-root', str(DATA_ROOT), '--config', 'configs/large.yaml', '--output', str(OUTPUT_ROOT)], check=True)
receipt = OUTPUT_ROOT / 'large_run_receipt.json'
receipt_data = json.loads(receipt.read_text())
assert receipt_data['is_google_colab']
assert len(receipt_data['profiles']) == 1
assert all(item['verified'] for item in receipt_data['profiles'])
print(json.dumps(receipt_data, indent=2))


## 5. Build and download the verified evidence ZIP

In [ ]:
export_root = PERSIST / 'evidence_package'
if export_root.exists():
    shutil.rmtree(export_root)
export_root.mkdir(parents=True, exist_ok=True)
shutil.copytree(OUTPUT_ROOT, export_root / 'artifacts', dirs_exist_ok=True)
shutil.copy2(dataset / 'dataset_manifest.json', export_root / 'dataset_manifest.json')
shutil.copy2(dataset / 'checksums.sha256', export_root / 'dataset_checksums.sha256')
shutil.copy2(OUTPUT_ROOT / 'large_run_receipt.json', export_root / 'large_run_receipt.json')
profile_receipt = receipt_data['profiles'][0]
package_manifest = {'schema_version': '1.0', 'profile': PROFILE, 'seed': SEED, 'git_commit': COMMIT, 'dataset_id': profile_receipt['dataset_id'], 'dataset_checksum': profile_receipt['dataset_checksum'], 'run_id': profile_receipt['run_id'], 'execution_platform': 'google_colab', 'files': {}}
for path in sorted(export_root.rglob('*')):
    if path.is_file():
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(chunk)
        package_manifest['files'][str(path.relative_to(export_root))] = digest.hexdigest()
package_manifest_path = export_root / 'evidence_package_manifest.json'
package_manifest_path.write_text(json.dumps(package_manifest, indent=2, sort_keys=True) + '\n')
archive_base = PERSIST / f'GraphTrust_large_{PROFILE}_{SEED}'
archive = Path(shutil.make_archive(str(archive_base), 'zip', PERSIST, 'evidence_package'))
archive_digest = hashlib.sha256()
with archive.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        archive_digest.update(chunk)
archive_sha256 = archive_digest.hexdigest()
package_manifest_digest = hashlib.sha256(package_manifest_path.read_bytes()).hexdigest()
download_receipt = {'archive': archive.name, 'archive_sha256': archive_sha256, 'package_manifest_sha256': package_manifest_digest, 'git_commit': COMMIT, 'profile': PROFILE, 'seed': SEED, 'dataset_id': profile_receipt['dataset_id'], 'dataset_checksum': profile_receipt['dataset_checksum'], 'run_id': profile_receipt['run_id'], 'execution_platform': 'google_colab'}
download_receipt_path = export_root / f'GraphTrust_large_{PROFILE}_{SEED}_download_receipt.json'
download_receipt_path.write_text(json.dumps(download_receipt, indent=2, sort_keys=True) + '\n')
print(download_receipt)
files.download(str(archive))
files.download(str(download_receipt_path))


## What to send back

Send the downloaded `GraphTrust_large_<profile>_<seed>.zip` and `GraphTrust_large_<profile>_<seed>_download_receipt.json`. Do not report runtime or memory numbers unless the receipt says `is_google_colab: true` and every profile says `verified: true`.